# Backfill author_registry.orcid from raw_orcid Conflicts

**Problem**: ~1.5M author entities have works carrying 2+ distinct `raw_orcid` values, indicating overmerged profiles. The current `author.orcid` in CreateAuthors uses "most recent work wins" which is unreliable.

**Approach**: For authors with conflicting raw_orcids that pass safeguards, set `author_registry.orcid` to the majority raw_orcid. CreateAuthors will then use this as the priority source for the author's ORCID.

**Safeguards**:
- Majority ORCID must have >= 3 works
- Majority must have >= 3x the works of all minorities combined

**oxjobs**: #110 split-author-profiles-on-orcid

### Cell 1: Diagnostic — Count authors with conflicting ORCIDs

In [ ]:
WITH author_orcids AS (
    SELECT
        CAST(SUBSTR(auth.author.id, 26) AS BIGINT) AS author_id,
        auth.raw_orcid,
        COUNT(*) AS work_count
    FROM openalex.works.openalex_works
    LATERAL VIEW POSEXPLODE(authorships) AS pos, auth
    WHERE auth.author.id IS NOT NULL
      AND auth.raw_orcid IS NOT NULL
      AND LENGTH(auth.raw_orcid) > 20
    GROUP BY 1, 2
),
author_stats AS (
    SELECT
        author_id,
        COUNT(*) AS distinct_orcids,
        MAX(work_count) AS majority_count,
        SUM(work_count) - MAX(work_count) AS minority_count
    FROM author_orcids
    GROUP BY author_id
    HAVING COUNT(*) >= 2
)
SELECT
    COUNT(*) AS authors_with_multiple_orcids,
    SUM(CASE WHEN majority_count >= 3 AND majority_count >= 3 * minority_count THEN 1 ELSE 0 END) AS authors_passing_safeguards
FROM author_stats

### Cell 2: Build majority orcid lookup

For each author with 2+ distinct raw_orcids passing safeguards, identifies the majority ORCID.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.orcid_majority_lookup AS
WITH author_orcids AS (
    SELECT
        CAST(SUBSTR(auth.author.id, 26) AS BIGINT) AS author_id,
        auth.raw_orcid,
        COUNT(*) AS work_count
    FROM openalex.works.openalex_works
    LATERAL VIEW POSEXPLODE(authorships) AS pos, auth
    WHERE auth.author.id IS NOT NULL
      AND auth.raw_orcid IS NOT NULL
      AND LENGTH(auth.raw_orcid) > 20
    GROUP BY 1, 2
)
SELECT
    author_id,
    MAX_BY(raw_orcid, work_count) AS majority_orcid,
    MAX(work_count) AS majority_count,
    SUM(work_count) - MAX(work_count) AS minority_count,
    COUNT(*) AS distinct_orcids
FROM author_orcids
GROUP BY author_id
HAVING COUNT(*) >= 2
   AND MAX(work_count) >= 3
   AND MAX(work_count) >= 3 * (SUM(work_count) - MAX(work_count))

### Cell 3: Inspect lookup table

In [ ]:
SELECT
    COUNT(*) AS total_authors,
    SUM(minority_count) AS total_minority_works
FROM openalex.authors.orcid_majority_lookup

In [ ]:
SELECT ol.author_id, ol.majority_orcid, ol.majority_count, ol.minority_count, ol.distinct_orcids,
       ar.display_name, ar.orcid AS current_registry_orcid
FROM openalex.authors.orcid_majority_lookup ol
LEFT JOIN openalex.authors.author_registry ar ON ol.author_id = ar.id
ORDER BY ol.minority_count DESC
LIMIT 30

### Cell 4: Backfill author_registry.orcid

Sets `orcid` on author_registry to the majority raw_orcid for authors with conflicting orcids that pass safeguards.

In [ ]:
MERGE INTO openalex.authors.author_registry AS target
USING openalex.authors.orcid_majority_lookup AS source
ON target.id = source.author_id
WHEN MATCHED THEN UPDATE SET
    target.orcid = source.majority_orcid,
    target.updated_date = current_timestamp()

### Cell 5: Verification

In [ ]:
-- Confirm registry orcids were set
SELECT COUNT(*) AS authors_with_registry_orcid
FROM openalex.authors.author_registry
WHERE orcid IS NOT NULL

In [ ]:
-- Spot-check: compare registry orcid to current openalex_authors orcid
SELECT ar.id AS author_id, ar.display_name, ar.orcid AS registry_orcid,
       oa.orcid AS current_profile_orcid,
       CASE WHEN ar.orcid = oa.orcid THEN 'match' ELSE 'different' END AS comparison
FROM openalex.authors.author_registry ar
JOIN openalex.authors.openalex_authors oa ON ar.id = oa.id
WHERE ar.orcid IS NOT NULL
ORDER BY RAND(42)
LIMIT 30

### Cell 6: Build minority works repair table

Identifies works assigned to these authors where the `raw_orcid` doesn't match the majority orcid. These are the wrongly-assigned works that need `author_id` NULLed out.

In [ ]:
CREATE OR REPLACE TABLE openalex.authors.orcid_minority_works_repair AS
SELECT
    w.id AS work_id,
    pos AS author_sequence,
    CAST(SUBSTR(auth.author.id, 26) AS BIGINT) AS author_id,
    auth.raw_orcid,
    m.majority_orcid
FROM openalex.works.openalex_works w
LATERAL VIEW POSEXPLODE(w.authorships) AS pos, auth
INNER JOIN openalex.authors.orcid_majority_lookup m
    ON CAST(SUBSTR(auth.author.id, 26) AS BIGINT) = m.author_id
WHERE auth.raw_orcid IS NOT NULL
  AND LENGTH(auth.raw_orcid) > 20
  AND auth.raw_orcid != m.majority_orcid

### Cell 7: Inspect minority works repair table

In [ ]:
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT work_id) AS distinct_works,
    COUNT(DISTINCT author_id) AS distinct_authors
FROM openalex.authors.orcid_minority_works_repair

In [ ]:
SELECT r.work_id, r.author_sequence, r.author_id, r.raw_orcid, r.majority_orcid,
       wa.raw_author_name,
       oa.display_name AS profile_display_name
FROM openalex.authors.orcid_minority_works_repair r
LEFT JOIN openalex.works.work_authors wa
    ON r.work_id = wa.work_id AND r.author_sequence = wa.author_sequence
LEFT JOIN openalex.authors.openalex_authors oa
    ON r.author_id = oa.id
ORDER BY RAND(42)
LIMIT 30

### Cell 8: NULL author_id on minority works

Safety guard: joins on `author_id` so we only NULL if the assignment hasn't changed since we built the repair table.

In [ ]:
MERGE INTO openalex.works.work_authors AS target
USING openalex.authors.orcid_minority_works_repair AS source
ON target.work_id = source.work_id
   AND target.author_sequence = source.author_sequence
   AND target.author_id = source.author_id
WHEN MATCHED THEN UPDATE SET
    target.author_id = NULL,
    target.updated_at = current_timestamp()

### Cell 9: Queue affected works for reprocessing

In [ ]:
INSERT INTO openalex.institutions.curated_work_ids_pending_sync (work_id, curated_ras, added_datetime)
SELECT DISTINCT
    work_id,
    'orcid_split_repair' AS curated_ras,
    current_timestamp() AS added_datetime
FROM openalex.authors.orcid_minority_works_repair
WHERE NOT EXISTS (
    SELECT 1 FROM openalex.institutions.curated_work_ids_pending_sync p
    WHERE p.work_id = openalex.authors.orcid_minority_works_repair.work_id
)

### Cell 10: Final verification

In [ ]:
-- Confirm affected rows are now NULL
SELECT COUNT(*) AS nulled_rows
FROM openalex.works.work_authors wa
JOIN openalex.authors.orcid_minority_works_repair r
    ON wa.work_id = r.work_id AND wa.author_sequence = r.author_sequence
WHERE wa.author_id IS NULL

In [ ]:
-- Confirm sync queue populated
SELECT COUNT(DISTINCT work_id) AS queued_works
FROM openalex.institutions.curated_work_ids_pending_sync
WHERE curated_ras = 'orcid_split_repair'